# CS781 Phase 4 (CJPE: court judgment prediction) — the full notebook: restore, the remaining required experiments, long-document study, ensemble + stackers, rule check

**What it does.** It reuses the cells of the v3 techniques notebook (data, Phase 1 loaders, adapters, trainer), **restores the GPT-2 and encoder runs you already finished** (so they are not retrained), runs only what is still missing, and then builds the best `prediction.json` it can justify on the dev split.

## Inputs it needs
1. **Your earlier Phase 4 notebook output** (the run that produced `P1_gpt2_full`, `P2_enc_full`, `P2_enc_full_W8`): Add Input > Notebook Output Files. It must hold `cjpe/ledger.csv`, `cjpe/runs/*` (per-window scores and the saved `model.pt` of P1 and P2: the distillation teachers) and `cjpe/submissions`. A backup dataset works instead if it exists.
2. **Phase 1 datasets:** the **encoder** (P3, the second encoder seed, the 8-window re-score of P2, and the P2 teacher), the **GPT-2** (P7 student, the P1 teacher, the 8-window re-score of P1), the **T5** (P4; its tokenizer is the one the earlier runs used) and the **BiLSTM** (P5; without it P5 skips and the rule check says that architecture is not covered). The tokenizer `legal_bpe_32k_mask.json` is found in them.
3. **Secrets** `HF_TOKEN` (the CJPE data, 703 MB, is downloaded again), `KAGGLE_USERNAME`, `KAGGLE_KEY` (the backup upload); **Internet ON**; **a GPU**. The credentials cell checks them first and nothing ever asks you to type anything.

## What runs
| cell | what | needs | time (T4, rough) |
|---|---|---|---|
| setup, credentials, data, models, adapters, trainer | the same code as the v3 notebook; the trainer can now score more windows at inference than it samples in training | the CJPE data + the Phase 1 tokenizer | ~15 min (download + tokenising) |
| restore | brings back the earlier runs and the ledger | the earlier notebook output | ~2 min |
| **P3** encoder **LoRMA+** (compulsory) | multiplicative low-rank adaptation | encoder | ~60 min |
| **P4** T5 | fp32 probe (LoRA, then full); a full run only if a probe beats chance; else a 45-minute capped LoRA run so the architecture is covered | T5 | 10-55 min |
| **P5** BiLSTM partial fine-tuning | embeddings + lowest layer frozen | BiLSTM | ~75 min |
| **P7** knowledge distillation | teachers P1 + P2 -> 6-layer GPT-2 student (+ control without distillation) | restored P1/P2 models, GPT-2 | ~120 min |
| X1 (optional) | a second encoder seed (an ensemble member, not a counted experiment) | encoder | ~55 min |
| P6 (optional, off by default) | GPT-2 IA3 | GPT-2 | ~60 min |
| 8 windows | re-scores the saved P1 and P2 with up to 8 windows (skipped where `_W8` already exists) | the models | ~3 min each |
| representation study | last window vs 4 vs 8 windows with paired bootstrap, from saved scores | none | seconds |
| **candidates, stackers, rule check** | best single, mean ensemble (one variant per checkpoint), S1 (learned model weights), S2 (a logistic regression over every window of every model + case length), the six-experiment rule check | the runs on disk | minutes (CPU) |
| backup, output | verified upload; output check | Kaggle key | ~5 min |
Choose which new runs happen with `RUN` in the settings cell. All of it is about 6-7 hours; the final cells work with whatever finished.

## The selection rules (stated, so you can check them)
- **Stackers are trained on the dev split, so they are scored out-of-fold** (5-fold cross-validation over dev cases) and a stacker replaces the plain ensemble only if its macro-F1 wins by 0.003, its AUC by 0.002 **and** its paired-bootstrap 95% interval is above 0. Otherwise the ensemble or best single run stays. Predictions are rate-matched to the dev positive rate (no test labels).
- **Experiments are distinct (Phase 1 model, technique) pairs.** Seeds, 8-window re-scores, controls and pilots are not experiments. The rule check needs six experiments, all four architectures, five technique families, LoRMA and distillation, and says exactly what is missing.

## Honest limits
Gains from the stackers, the extra members and 8 windows are my estimates (a few thousandths to ~0.015 AUC), not measurements; the comparison table printed by the final cell is the evidence. T5 has not learned in earlier runs; P4 is built so the rules can be met even if it does not, but then it is weak and excluded from the ensemble. The dev split has 1,988 cases, so one standard error of the macro-F1 is about 0.01.


In [ ]:
# ================= CELL 0 — setup =================
import os, sys, re, json, math, time, random, glob, gc, copy, hashlib, pickle, getpass, zipfile
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda"
WORK = Path("/kaggle/working/cjpe"); (WORK / "runs").mkdir(parents=True, exist_ok=True); (WORK / "cache").mkdir(exist_ok=True); SUBDIR = WORK / "submissions"; SUBDIR.mkdir(exist_ok=True)
LEDGER = WORK / "ledger.csv"
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu (training cells need a GPU)", "| work dir:", WORK)


In [ ]:
# ===== CREDENTIALS CHECK (run it FIRST; it never asks you to type anything): reads HF_TOKEN, KAGGLE_USERNAME and KAGGLE_KEY from Kaggle Secrets, verifies them NOW, and stops early with the exact fix =====   [CELL 1]
# Needs: Internet ON, and the three Secrets added under Add-ons > Secrets with the 'Attached' toggle ON for THIS notebook (a committed run only sees attached secrets).
import os, glob, subprocess, shutil
from pathlib import Path
REQUIRE_HF, REQUIRE_KAGGLE = True, True        # True = stop now if that credential does not work; False = go on without it (no Hugging Face download / no backup upload)
HF_REPO = "Exploration-Lab/CS-781-Capstone"
def get_secret(name):
    """environment variable first, then the Kaggle Secret of the same name; quotes and spaces stripped; never printed"""
    v = os.environ.get(name)
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        except Exception: v = None
    return (v or "").strip().strip("'\"") or None
problems, report = [], []
# ---- 1) Hugging Face token: only needed when the CJPE json files are not already on disk or attached as a dataset ----
have_local = all(glob.glob(f"/kaggle/input/**/{n}", recursive=True) or glob.glob(f"/kaggle/working/**/{n}", recursive=True) for n in ("train.json", "dev.json", "test.json"))
hf = get_secret("HF_TOKEN"); HF_OK = False
if have_local: report.append("Hugging Face token : not needed (the CJPE files are already on disk / attached)"); HF_OK = True
elif not hf: report.append("Hugging Face token : MISSING (no HF_TOKEN secret or environment variable)"); problems.append("HF_TOKEN is missing: add a Secret named HF_TOKEN (a token of the account that was approved for the dataset), or attach the CJPE files as a Kaggle dataset")
else:
    try:
        from huggingface_hub import HfApi; api = HfApi(); me = api.whoami(token=hf); api.auth_check(HF_REPO, repo_type="dataset", token=hf); HF_OK = True; os.environ["HF_TOKEN"] = hf
        report.append(f"Hugging Face token : OK (user '{me.get('name')}', access to {HF_REPO} confirmed, {len(hf)} characters)")
    except Exception as e:
        code = getattr(getattr(e, "response", None), "status_code", None)
        why = "Hugging Face does not recognise it: it is wrong, expired or revoked (create a new classic 'Read' token and update the Secret)" if code == 401 else "the token is valid but this account has no approved access to the gated dataset" if code == 403 else f"{type(e).__name__}: {str(e)[:100]}"
        report.append(f"Hugging Face token : REFUSED (HTTP {code}): {why}"); problems.append(f"HF_TOKEN does not work: {why}")
# ---- 2) Kaggle username + key: needed to upload the backup dataset ----
ku, kk = get_secret("KAGGLE_USERNAME"), get_secret("KAGGLE_KEY"); KAGGLE_OK = False
if not (ku and kk): report.append(f"Kaggle credentials  : MISSING ({'KAGGLE_USERNAME' if not ku else ''}{' and ' if not (ku or kk) else ''}{'KAGGLE_KEY' if not kk else ''} not found)"); problems.append("KAGGLE_USERNAME / KAGGLE_KEY are missing: add them as Secrets (the LEGACY API key from kaggle.com/settings > API)")
elif not shutil.which("kaggle"): report.append("Kaggle credentials  : cannot be verified (the kaggle command is not installed in this image)"); problems.append("the kaggle command is not installed, so the backup upload cannot run")
else:
    try:
        r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=120, env={**os.environ, "KAGGLE_USERNAME": ku, "KAGGLE_KEY": kk})
        if r.returncode == 0: KAGGLE_OK = True; os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = ku, kk; report.append(f"Kaggle credentials  : OK (user '{ku}', the API accepted the key)")
        else: report.append(f"Kaggle credentials  : REFUSED ({(r.stdout + r.stderr).strip()[:100]})"); problems.append("the Kaggle username/key were refused: check that KAGGLE_KEY is the LEGACY API key and that Internet is ON")
    except Exception as e: report.append(f"Kaggle credentials  : not verified ({type(e).__name__})"); problems.append(f"the Kaggle credentials could not be checked ({type(e).__name__}): is Internet ON?")
print("\n".join(report))
blocking = [p for p in problems if (p.startswith("HF_TOKEN") and REQUIRE_HF) or (not p.startswith("HF_TOKEN") and REQUIRE_KAGGLE)]
if blocking: raise RuntimeError("STOPPING EARLY (before any long step):\n  - " + "\n  - ".join(blocking) + "\nFix: Add-ons > Secrets > add the missing secret(s) and switch 'Attached' ON for this notebook, then run again. (Set REQUIRE_HF / REQUIRE_KAGGLE = False at the top of this cell to continue without them.)")
if problems: print("\nContinuing without:", "; ".join(problems))
print("\nCredentials ready." if not problems else "")


In [ ]:
# ================= CELL 2 — CJPE data: locate (or download), detect the schema, tokenise ONLY what is needed (the end of each case), cache =================
HF_REPO = "Exploration-Lab/CS-781-Capstone"          # gated: Phase_4_CJPE/Train/train.json (737 MB), Train/dev.json (80 MB), Test/test.json (35 MB)
TAIL_MAX = 4096                                      # tokens kept from the END of every case (the end of a judgment is the most informative part)
TEXT_KEYS = ("text", "document", "case_text", "content", "judgment", "doc", "full_text", "case"); LABEL_KEYS = ("label", "labels", "decision", "outcome", "y", "gold", "judgement", "verdict")
ID_KEYS = ("id", "case_id", "doc_id", "document_id", "name", "file", "filename", "index")

def locate():
    got = {}
    for sp in ("train", "dev", "test"):
        c = [p for root in SEARCH_ROOTS for p in glob.glob(f"{root}/**/{sp}.json", recursive=True) if re.search(r"cjpe|phase[_ -]?4", p, re.I)]
        if c: got[sp] = max(c, key=os.path.getsize)
    return got
paths = locate()
if len(paths) < 3:
    print("CJPE files not found locally -> downloading from Hugging Face (accept the dataset conditions on its page first)")
    from huggingface_hub import HfApi, snapshot_download
    def hf_token_ok(tk):
        """True only if Hugging Face accepts the token AND the gated dataset is readable with it (a wrong token used to fail later with a confusing 401)"""
        try:
            api = HfApi(); me = api.whoami(token=tk); api.auth_check(HF_REPO, repo_type="dataset", token=tk); print(f"   accepted: user '{me.get('name')}', access to {HF_REPO} OK"); return True
        except Exception as e:
            code = getattr(getattr(e, "response", None), "status_code", None); print(f"   REFUSED (HTTP {code}, {type(e).__name__}): " + ("Hugging Face does not recognise this token" if code == 401 else "the token is valid but this account has no approved access to the dataset"
                  if code == 403 else str(e)[:120])); return False
    cands = []
    if os.environ.get("HF_TOKEN"): cands.append(("environment variable HF_TOKEN", os.environ["HF_TOKEN"]))
    try:
        from kaggle_secrets import UserSecretsClient; cands.append(("Kaggle secret HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN")))
    except Exception: pass
    token = None
    for src_, tk in cands:
        tk = (tk or "").strip().strip("'\""); print(f"trying the token from the {src_} ({len(tk)} characters)")
        if tk and hf_token_ok(tk): token = tk; break
        print(f"   -> not usable; if this is a Kaggle secret it is probably stale: update or delete it")
    assert token, "no working Hugging Face token and the CJPE files are not attached. This cell never asks you to type anything. Fix: add a Kaggle Secret named HF_TOKEN (Add-ons > Secrets, Attached ON for this notebook) holding a token of the account that was approved for the dataset, or attach the CJPE files as a Kaggle dataset"
    os.environ["HF_TOKEN"] = token
    local = snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["Phase_4_CJPE/*"], token=token, local_dir=str(WORK / "hf")); SEARCH_ROOTS.append(local); paths = locate()
assert len(paths) == 3, f"need train/dev/test json, found {paths}"
print({k: f"{os.path.getsize(v)/2**20:.0f} MiB  {v}" for k, v in paths.items()})

def normalise(obj):
    if isinstance(obj, dict):
        for wk in ("data", "documents", "examples", "cases", "items", "train", "dev", "test", "validation"):
            if wk in obj and isinstance(obj[wk], (list, dict)) and len(obj) <= 4: return normalise(obj[wk])
        out = []
        for k, v in obj.items():
            if isinstance(v, dict): r = dict(v); r.setdefault("id", k)
            elif isinstance(v, str): r = {"id": k, "text": v}
            else: continue
            out.append(r)
        return out
    return [r for r in obj if isinstance(r, dict)]
def detect(recs):
    keys = list(recs[0].keys()); strlen = {k: np.mean([len(str(r.get(k, ""))) for r in recs[:200]]) for k in keys if isinstance(recs[0].get(k), str)}
    tk = next((k for k in TEXT_KEYS if k in keys), None) or (max(strlen, key=strlen.get) if strlen else None); lk = next((k for k in LABEL_KEYS if k in keys), None)
    if lk is None:
        for k in keys:
            if k == tk: continue
            vals = {str(r.get(k)) for r in recs[:2000] if r.get(k) is not None}
            if 1 < len(vals) <= 5 and k not in ID_KEYS: lk = k; break
    return tk, lk, next((k for k in ID_KEYS if k in keys), None)
def binary_map(vals):
    s = {str(v).strip().lower() for v in vals if v is not None}
    if s <= {"0", "1", "0.0", "1.0", "true", "false"}: return lambda v: int(str(v).strip().lower() in ("1", "1.0", "true"))
    raise ValueError(f"label values {sorted(s)[:10]} are not 0/1: tell me what they mean (for example accepted/rejected) before training")

from tokenizers import Tokenizer
tpaths = sorted(glob.glob("/kaggle/input/**/legal_bpe_32k_mask.json", recursive=True)) or sorted(glob.glob("/kaggle/input/**/legal_bpe_32k*.json", recursive=True)) or sorted(glob.glob("/kaggle/working/**/legal_bpe_32k*.json", recursive=True))
assert tpaths, "attach the tokenizer file legal_bpe_32k_mask.json (it is inside your Phase 1 dataset)"
tok = Tokenizer.from_file(tpaths[0]); EOS, BOS, PAD = tok.token_to_id("[EOS]"), tok.token_to_id("[BOS]"), tok.token_to_id("[PAD]"); print("tokenizer:", tpaths[0], "| vocabulary", tok.get_vocab_size())

def prepare(split, path, bs=256):
    cache = WORK / "cache" / f"{split}.pkl"
    if cache.exists():
        d = pickle.load(open(cache, "rb")); print(f"[{split}] loaded cache: {len(d['tails'])} cases"); return d
    print(f"[{split}] reading {path} ..."); obj = json.load(open(path, encoding="utf-8")); recs = normalise(obj); del obj; gc.collect()
    tk, lk, ik = detect(recs); lmap = binary_map([r.get(lk) for r in recs[:5000]]) if lk else None
    print(f"[{split}] {len(recs)} cases | text field '{tk}' | label field '{lk}' | id field '{ik}'")
    tails, lens, labels, rids = [], [], [], []
    for i in range(0, len(recs), bs):
        chunk = recs[i:i + bs]
        for e in tok.encode_batch([str(r.get(tk, "")) for r in chunk], add_special_tokens=False):
            lens.append(len(e.ids)); tails.append(np.asarray((e.ids + [EOS])[-TAIL_MAX:], dtype=np.int32))
        labels += [lmap(r.get(lk)) if lmap else -1 for r in chunk]; rids += [r.get(ik, i + j) if ik else i + j for j, r in enumerate(chunk)]
        for r in chunk: r.clear()
    if split == "test":
        if lk and any(l >= 0 for l in labels): print("[test] NOTE: test.json contains a label field. The task says test labels are private, so they are IGNORED here and never used for anything.")
        labels = [-1] * len(labels)
    d = dict(tails=tails, lens=np.asarray(lens), labels=np.asarray(labels), rids=rids, keys=dict(text=tk, label=lk, id=ik)); pickle.dump(d, open(cache, "wb"), protocol=4); del recs; gc.collect(); return d
DATA = {sp: prepare(sp, p) for sp, p in paths.items()}
DATA["test"]["labels"] = np.full(len(DATA["test"]["tails"]), -1)       # test labels are private: blank them even if a cache from an earlier run held them
for sp, d in DATA.items():
    y = d["labels"]; print(f"[{sp}] cases {len(d['tails']):6d} | tokens p50/p90/p99: {int(np.percentile(d['lens'],50))}/{int(np.percentile(d['lens'],90))}/{int(np.percentile(d['lens'],99))} | label counts: {dict(Counter(y.tolist())) if (y>=0).any() else 'none (test)'}")
assert len(set(map(str, DATA["test"]["rids"]))) == len(DATA["test"]["rids"]), "duplicate test ids"
ytr = DATA["train"]["labels"]; print(f"majority-class baseline macro-F1 on dev would be about {0.5 * (2*max(np.mean(DATA['dev']['labels']==0), np.mean(DATA['dev']['labels']==1)) / (1+max(np.mean(DATA['dev']['labels']==0), np.mean(DATA['dev']['labels']==1)))):.3f}")


In [ ]:
# ================= CELL 3 — load Phase 1 models (all four architectures; the old 3,000-step GPT-2 is never used), document classifier, chunked batching =================
al_paths = sorted(glob.glob("/kaggle/input/**/arch_loaders.py", recursive=True)) or sorted(glob.glob("/kaggle/working/**/arch_loaders.py", recursive=True))
assert al_paths, "attach the dataset/notebook output that contains arch_loaders.py (written by cell 8 of the pre-training notebooks)"
sys.path.insert(0, str(Path(al_paths[0]).parent)); import arch_loaders as AL

MODEL_PATHS = {}                                    # filled automatically below; you can also set entries by hand, e.g. MODEL_PATHS["bilstm"] = "/kaggle/input/.../final_model.pt"
def discover_models():
    found = {}
    for p in sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)):
        if os.path.getsize(p) < 50e6: continue
        base = os.path.basename(p)
        if base != "final_model.pt": continue                # only the NEW Phase 1 models: the old 3,000-step checkpoint is deliberately never used
        try: ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: continue
        if not isinstance(ck, dict) or "model_config" not in ck or "model_state_dict" not in ck: continue
        mc = ck["model_config"]; fam = mc.get("family")
        if fam is None: continue                              # a checkpoint without a family entry is an old-format model: skipped
        key = "gpt2_new" if fam == "gpt2" else fam
        if key not in found: found[key] = p
    return found
MODEL_PATHS.update({k: v for k, v in discover_models().items() if k not in MODEL_PATHS})
print("Phase 1 models found:", {k: v for k, v in MODEL_PATHS.items()}, "\n  missing (pre-train them first):", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k not in MODEL_PATHS])

def load_backbone_any(path):
    """like AL.load_backbone, but also accepts the OLD Phase 1 checkpoint (its model_config has no 'family' entry)"""
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = dict(ck["model_config"]); mc.setdefault("family", "gpt2")
    full = AL.build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True); f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    del ck; return AL.Backbone(f, inner, embed), mc["hidden_size"], AL.CAUSAL[f], mc["context_length"], f

class DocClassifier(nn.Module):
    """Encodes K chunks of a case (the last K*L tokens, right-aligned) with a Phase 1 backbone, pools each chunk, combines the chunks with attention pooling
       + the final chunk, and classifies. K=1 is the plain 'last L tokens' model. Chunks that are empty (short cases) are skipped, never fed to the backbone."""
    def __init__(self, backbone, hidden, causal, K, dropout=0.1, pool="last"):
        super().__init__(); self.backbone, self.hidden, self.causal, self.K, self.pool = backbone, hidden, causal, K, pool
        self.chunk_pos = nn.Parameter(torch.zeros(K, hidden)); self.proj = nn.Linear(hidden, hidden); self.att = nn.Linear(hidden, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 2))
    def encode_chunks(self, ids, mask):
        B, K, L = ids.shape; valid = mask.sum(-1) > 0; fi, fm = ids[valid], mask[valid]; T = int(fm.sum(1).max()); fi, fm = fi[:, :T], fm[:, :T]
        h = self.backbone(input_ids=fi, attention_mask=fm).last_hidden_state
        mm = fm.unsqueeze(-1).to(h.dtype); mean = (h * mm).sum(1) / mm.sum(1).clamp(min=1)
        if self.causal:
            last = h[torch.arange(h.shape[0], device=h.device), fm.sum(1) - 1]; pooled = last if self.pool == "last" else 0.5 * (last + mean)      # pool='meanlast': average of the last-token and mean-pooled vectors
        else: pooled = mean
        out = pooled.new_zeros(B, K, self.hidden); out[valid] = pooled; return out, valid
    def forward(self, ids, mask):
        x, valid = self.encode_chunks(ids, mask); x = x + self.chunk_pos.to(x.dtype)
        s = self.att(torch.tanh(self.proj(x))).squeeze(-1).float().masked_fill(~valid, -1e4); w = torch.softmax(s, -1).to(x.dtype)
        return self.head(torch.cat([(w.unsqueeze(-1) * x).sum(1), x[:, -1]], -1))

def make_batch(tails, idx, K, L):
    """last K*L tokens of each case, split RIGHT-aligned into K chunks of <= L tokens (chunk K-1 is the very end); right-padded; returns ids/mask [B,K,L]"""
    B = len(idx); ids = np.full((B, K, L), PAD, dtype=np.int64); mask = np.zeros((B, K, L), dtype=np.int64)
    for b, i in enumerate(idx):
        t = tails[i][-K * L:]; n = len(t)
        for k in range(K - 1, -1, -1):
            end = n - (K - 1 - k) * L
            if end <= 0: break
            seg = t[max(0, end - L):end]; ids[b, k, :len(seg)] = seg; mask[b, k, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)

def set_backbone_dropout(backbone, p):
    """turn on dropout inside the backbone for fine-tuning (works where the model has nn.Dropout modules, e.g. GPT-2 and T5; a no-op for the others). Returns how many modules were changed."""
    n = 0
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p; n += 1
    return n

# ---- windows: window k = the k-th block of L tokens counted back from the end of the case (k=0 is the very end) ----
def n_windows(tail_len, L, W): return max(1, min(W, -(-tail_len // L)))
def get_window(tail, k, L):
    n = len(tail); end = n - k * L
    return tail[-L:] if end <= 0 else tail[max(0, end - L):end]
def make_window_batch(tails, idx, ks, L):
    B = len(idx); ids = np.full((B, 1, L), PAD, dtype=np.int64); mask = np.zeros((B, 1, L), dtype=np.int64)
    for b, (i, k) in enumerate(zip(idx, ks)):
        seg = get_window(tails[i], int(k), L); ids[b, 0, :len(seg)] = seg; mask[b, 0, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)

def truncate_layers(bb, n):
    """keep only the first n transformer layers (initialises a smaller student from the same Phase 1 weights); implemented for the GPT-2 family"""
    assert bb.family == "gpt2", "layer truncation is implemented for the GPT-2 family only"
    bb.inner.h = nn.ModuleList(list(bb.inner.h)[:n]); bb.inner.config.n_layer = n; return bb

def refresh_models(verbose=True):
    """re-scan /kaggle/input for Phase 1 models (for example after you attached another dataset to the running session)"""
    new = {k: v for k, v in discover_models().items() if k not in MODEL_PATHS}; MODEL_PATHS.update(new)
    if verbose and new: print("newly found Phase 1 models:", new)
    return sorted(MODEL_PATHS)


In [ ]:
# ================= CELL 4 — adaptation techniques: full, partial (layer-wise freezing), LoRA, rsLoRA, LoRMA+, LoRMA-pi, IA3 =================
from transformers.pytorch_utils import Conv1D
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A) @ self.B * self.scale
class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B = 0 at init  (= W0 x + (alpha/r) B A (W0 x))
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t()) @ self.B.t()) * self.scale
class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0; A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A); i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)
class _Wrap:
    """expose the wrapped layer's weight / bias: some Hugging Face modules read them directly (T5's feed-forward dtype check does)"""
    @property
    def weight(self): return self.base.weight
    @property
    def bias(self): return self.base.bias
def proj_dims(m):
    return (m.in_features, m.out_features) if isinstance(m, nn.Linear) else tuple(m.weight.shape)      # HF Conv1D weight is [in, out]
class FusedQV(_Wrap, nn.Module):                  # wraps a FUSED q|k|v projection (GPT-2 Conv1D c_attn, ModernBERT Linear Wqkv); LoRA / LoRMA adapt q and v, IA3 rescales k and v
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); nx, nf = proj_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora": self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi": self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        elif kind == "ia3": self.lk, self.lv = nn.Parameter(torch.ones(d)), nn.Parameter(torch.ones(d))
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora": q, v = q + self.q(x), v + self.v(x)
        elif self.kind.startswith("lorma"): q, v = self.q(q), self.v(v)
        else: k, v = k * self.lk, v * self.lv
        return torch.cat([q, k, v], -1)
class IA3Input(_Wrap, nn.Module):                 # IA3 on the FFN: rescale the activation entering the down-projection
    def __init__(self, base): super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(proj_dims(base)[0]))
    def forward(self, x): return self.base(x * self.l)
class LinAdapt(_Wrap, nn.Module):                 # the same adapters for separate nn.Linear projections (T5's q, k, v)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        elif kind == "ia3": self.l = nn.Parameter(torch.ones(do))
    def forward(self, x):
        y = self.base(x)
        if self.kind == "lora": return y + self.op(x)
        if self.kind == "ia3": return y * self.l
        return self.op(y)
def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw)); n += 1
    assert n > 0, f"no nn.Linear named {names} found in the backbone"
def freeze_lower(model, frac=0.5):
    """partial / layer-wise fine-tuning: freeze the embeddings and the lowest `frac` of the layers, train the rest"""
    inner, fam = model.backbone.inner, model.backbone.family
    for p in inner.parameters(): p.requires_grad = True
    if fam == "bilstm":
        n = inner.layers; k = int(n * frac)
        for p in inner.emb.parameters(): p.requires_grad = False
        for name, p in list(inner.fwd.named_parameters()) + list(inner.bwd.named_parameters()):
            if int(name.split("_l")[-1]) < k: p.requires_grad = False
        return
    if fam == "gpt2": layers, pre = list(inner.h), [inner.wte, inner.wpe]
    elif fam == "modernbert": layers, pre = list(inner.layers), [inner.embeddings]
    elif fam == "t5": layers, pre = list(inner.block), [inner.embed_tokens]
    else: raise ValueError(fam)
    k = int(len(layers) * frac)
    for mod in pre:
        for p in mod.parameters(): p.requires_grad = False
    for lay in layers[:k]:
        for p in lay.parameters(): p.requires_grad = False
METHODS = ("full", "partial", "lora", "rslora", "lorma_plus", "lorma_pi", "ia3")
ADAPTER_LR = {"lora": 5e-4, "rslora": 5e-4, "lorma_plus": 5e-4, "lorma_pi": 2e-4, "ia3": 3e-3}        # starting points; use lr_pilot(..., method=...) to check them
def apply_method(model, method, hp=None):
    assert method in METHODS, method; hp = hp or {}
    if method == "full":
        for p in model.backbone.parameters(): p.requires_grad = True
        return model
    if method == "partial": freeze_lower(model, hp.get("frac", 0.5)); return model
    fam, inner = model.backbone.family, model.backbone.inner
    for p in model.backbone.parameters(): p.requires_grad = False
    kind = {"lora": "lora", "rslora": "lora", "lorma_plus": "lorma_plus", "lorma_pi": "lorma_pi", "ia3": "ia3"}[method]; kw = dict(r=hp.get("r", 16), alpha=hp.get("alpha", 32), drop=hp.get("adapter_dropout", 0.05), rs=(method == "rslora"))
    if fam == "gpt2":
        for blk in inner.h:
            assert isinstance(blk.attn.c_attn, Conv1D), "unexpected c_attn type in this transformers version"; blk.attn.c_attn = FusedQV(blk.attn.c_attn, kind, **kw)
            if kind == "ia3": blk.mlp.c_proj = IA3Input(blk.mlp.c_proj)
    elif fam == "modernbert":
        for lay in inner.layers:
            assert isinstance(lay.attn.Wqkv, nn.Linear); lay.attn.Wqkv = FusedQV(lay.attn.Wqkv, kind, **kw)
            if kind == "ia3": lay.mlp.Wo = IA3Input(lay.mlp.Wo)
    elif fam == "t5":
        wrap_linear_targets(inner, kind, ("k", "v") if kind == "ia3" else ("q", "v"), **({} if kind == "ia3" else kw))
        if kind == "ia3":
            for _, parent in list(inner.named_modules()):
                if "DenseGatedActDense" in type(parent).__name__ or "DenseReluDense" in type(parent).__name__: parent.wo = IA3Input(parent.wo)
    else: raise NotImplementedError(f"'{method}' needs attention/linear layers: the {fam} model supports only 'full' and 'partial'")
    return model


In [ ]:
# ================= CELL 5 — trainer: any adaptation method, optional distillation, multi-window fine-tuning, EMA, multi-window inference, rate-matched threshold, submissions =================
import contextlib
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
def amp_ctx():
    if not AMP: return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)
DEFAULT_LR = {"gpt2": 3e-5, "modernbert": 5e-5, "t5": 5e-5, "bilstm": 5e-4}      # gpt2: best of 3e-5 / 1e-4 / 3e-4 in your pilot (the lowest won); the others are untested starting points: use lr_pilot() below
POLICY = "rate_matched"       # test-set decision rule: "rate_matched" (flag the top share of cases as 1, share = the dev positive rate; uses NO test labels) or "dev" (the threshold tuned on dev)
AGG_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]      # recency decay for averaging window scores: weight of window k is a**k (a=0: last window only, a=1: plain mean)

def prf(y, pred):
    f1s, ps, rs = [], [], []
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum()); fn = int(((pred != c) & (y == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0; f = 2 * p * r / (p + r) if p + r else 0.0; ps.append(p); rs.append(r); f1s.append(f)
    return dict(macro_f1=float(np.mean(f1s)), macro_p=float(np.mean(ps)), macro_r=float(np.mean(rs)), f1_0=f1s[0], f1_1=f1s[1], p1=ps[1], r1=rs[1])
def best_threshold(y, prob):
    best_f, best_t = -1.0, 0.5
    for t in sorted(np.arange(0.02, 0.981, 0.01), key=lambda t: abs(t - 0.5)):
        f = prf(y, (prob >= t).astype(int))["macro_f1"]
        if f > best_f + 1e-12: best_f, best_t = f, float(t)
    return best_t, best_f
def roc_auc(y, s):
    r = pd.Series(s).rank().values; n1 = int((y == 1).sum()); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 and n0 else float("nan")
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
def rm_predict(score, prior):
    """rate matching: the top round(prior*N) scores become class 1. Needs only the scores and the class prior (from dev), never labels of the set being predicted."""
    k = int(round(prior * len(score))); pred = np.zeros(len(score), dtype=int); pred[np.argsort(-score, kind="stable")[:k]] = 1; return pred

@torch.no_grad()
def predict_windows(model, tails, L, W, tok_budget=16384):
    """margin (logit1 - logit0) of each of the last W windows of every case; NaN where the case has fewer windows"""
    model.eval(); n = len(tails); nw = np.array([n_windows(len(t), L, W) for t in tails]); Z = np.full((n, W), np.nan, dtype=np.float32); mb = max(1, tok_budget // L)
    for k in range(W):
        idx_k = np.where(nw > k)[0]
        if len(idx_k) == 0: continue
        order = idx_k[np.argsort([-len(tails[i]) for i in idx_k], kind="stable")]
        for s in range(0, len(order), mb):
            sel = order[s:s + mb]; ids, mask = make_window_batch(tails, sel, [k] * len(sel), L)
            with amp_ctx(): lg = model(ids.to(device), mask.to(device)).float()
            Z[sel, k] = (lg[:, 1] - lg[:, 0]).cpu().numpy()
    return Z
def aggregate(Z, a):
    w = np.array([a ** k for k in range(Z.shape[1])], dtype=np.float32); den = np.where(np.isnan(Z), 0.0, w[None, :]).sum(1)
    return np.nansum(Z * w[None, :], axis=1) / den

class EMA:
    """exponential moving average of the weights; evaluated instead of the raw weights (smoother, usually better at the end of fine-tuning)"""
    def __init__(self, model, decay):
        self.params = [p for p in model.parameters() if p.requires_grad]; self.shadow = [p.detach().clone() for p in self.params]; self.decay = decay; self.backup = None
    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1 + step) / (10 + step)); torch._foreach_mul_(self.shadow, d); torch._foreach_add_(self.shadow, [p.detach() for p in self.params], alpha=1 - d)
    @torch.no_grad()
    def apply(self):
        self.backup = [p.detach().clone() for p in self.params]
        for p, s in zip(self.params, self.shadow): p.copy_(s)
    @torch.no_grad()
    def restore(self):
        for p, b in zip(self.params, self.backup): p.copy_(b)
        self.backup = None

def to_py(x): return x.item() if hasattr(x, "item") else x
def write_submission(path, rids, preds): json.dump([{"id": to_py(r), "pred": int(p)} for r, p in zip(rids, preds)], open(path, "w"))
def validate_submission(path, rids):
    sub = json.load(open(path)); assert isinstance(sub, list) and len(sub) == len(rids), f"{path}: expected {len(rids)} predictions, found {len(sub)}"
    for s, r in zip(sub, rids):
        assert set(s) == {"id", "pred"}, f"bad keys {set(s)}"; assert type(s["pred"]) is int and s["pred"] in (0, 1), f"pred must be the integer 0 or 1, got {s['pred']!r}"; assert s["id"] == to_py(r), f"id mismatch {s['id']!r} vs {to_py(r)!r}"
    assert len({str(s['id']) for s in sub}) == len(sub), "duplicate ids"; return True

def run_experiment(name, model_key, method="full", hp=None, n_layers=None, kd=None, L=512, W=4, epochs=3.0, max_minutes=None, lr=None, head_lr=1e-3, bs=16, wd=0.01, warmup=0.06, p_tail=0.5, ema_decay=0.999,
                   backbone_dropout=0.0, pool="last", dropout=0.1, seed=SEED, max_train_docs=None, dev_cap=None, test_cap=None, evals=4, write=True, policy=None, save_model=False, control=False, W_infer=None):
    """method: full | partial | lora | rslora | lorma_plus | lorma_pi | ia3 ;  kd=dict(path=..., T=3.0, alpha=0.5) distils from the teacher margins made by make_teacher() ;  n_layers=6 builds a half-depth GPT-2 student ;  control=True marks an ablation run that is never one of the six submissions"""
    t_start = time.time(); seed_all(seed); rng = np.random.default_rng(seed); policy = policy or POLICY
    assert model_key in MODEL_PATHS, f"'{model_key}' is not available: pre-train it first (found: {list(MODEL_PATHS)})"
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS[model_key]); L = min(L, ctx)
    if n_layers: truncate_layers(bb, n_layers)
    lr = lr or (DEFAULT_LR[fam] if method in ("full", "partial") else ADAPTER_LR[method]); technique = "kd" if kd else method
    nd = set_backbone_dropout(bb, backbone_dropout) if backbone_dropout else 0
    model = DocClassifier(bb, hidden, causal, 1, dropout, pool); apply_method(model, method, hp); model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters()); n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tr, dv, te = DATA["train"], DATA["dev"], DATA["test"]
    tr_idx = np.arange(len(tr["tails"])) if not max_train_docs else rng.choice(len(tr["tails"]), min(max_train_docs, len(tr["tails"])), replace=False)
    dv_idx = np.arange(len(dv["tails"])) if not dev_cap else rng.choice(len(dv["tails"]), min(dev_cap, len(dv["tails"])), replace=False); te_idx = np.arange(len(te["tails"])) if not test_cap else np.arange(min(test_cap, len(te["tails"])))
    dv_tails = [dv["tails"][i] for i in dv_idx]; y_dev = dv["labels"][dv_idx]; te_tails = [te["tails"][i] for i in te_idx]; prior = float(np.mean(dv["labels"] == 1))
    sub_n = min(1500, len(dv_idx)); sub = rng.choice(len(dv_idx), sub_n, replace=False); sub_tails = [dv_tails[i] for i in sub]; y_sub = y_dev[sub]
    nwin = np.array([n_windows(len(t), L, W) for t in tr["tails"]])
    cnt = np.bincount(tr["labels"][tr_idx], minlength=2).astype(float); cw = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    mb = max(1, 8192 // L); accum = max(1, math.ceil(bs / mb)); docs_per_step = mb * accum
    bb_train = [p for p in model.backbone.parameters() if p.requires_grad]; groups = ([{"params": bb_train, "lr": lr}] if bb_train else []) + [{"params": [p for n, p in model.named_parameters() if not n.startswith("backbone.")], "lr": head_lr}]
    tm = None
    if kd:
        tm = np.load(kd["path"]).astype(np.float32); assert len(tm) == len(tr["tails"]), "teacher margins do not match the training set: run make_teacher() on the full training set"
    opt = torch.optim.AdamW(groups, weight_decay=wd); base_lrs = [g["lr"] for g in groups]; scaler = torch.amp.GradScaler("cuda", enabled=AMP and not USE_BF16); ema = EMA(model, ema_decay)
    def sample_ks(idx):
        ks = np.zeros(len(idx), dtype=int)
        for j, i in enumerate(idx):
            if nwin[i] > 1 and rng.random() > p_tail: ks[j] = rng.integers(1, nwin[i])      # an earlier window, with probability 1 - p_tail
        return ks
    def micro(ids_idx):
        ids, mask = make_window_batch(tr["tails"], ids_idx, sample_ks(ids_idx), L); y = torch.from_numpy(tr["labels"][ids_idx]).to(device)
        with amp_ctx(): logits = model(ids.to(device), mask.to(device))
        logits = logits.float(); ce = F.cross_entropy(logits, y, weight=cw)
        if tm is None: return ce
        T_, a_ = kd.get("T", 3.0), kd.get("alpha", 0.5); pt = torch.sigmoid(torch.from_numpy(tm[ids_idx]).to(device) / T_); td = torch.stack([1 - pt, pt], 1)      # teacher P(class 1), softened by the temperature
        kl = (td * (torch.log(td.clamp_min(1e-8)) - F.log_softmax(logits / T_, -1))).sum(1).mean() * T_ * T_
        return a_ * ce + (1 - a_) * kl
    model.train(); times = []
    for j in range(4):                                                            # throughput probe (forward + backward only, no optimizer step); the first is warm-up
        t0 = time.time(); loss = micro(rng.choice(tr_idx, mb)); scaler.scale(loss).backward(); opt.zero_grad(set_to_none=True)
        if device.type == "cuda": torch.cuda.synchronize()
        times.append(time.time() - t0)
    micro_s = float(np.mean(times[1:])); sec_step = micro_s * accum; per_doc_fwd = micro_s / mb / 3.0
    steps = max(10, int(math.ceil(epochs * len(tr_idx) / docs_per_step))); W_inf = W_infer or W; eval_s = ((len(dv_tails) + len(te_tails)) * W_inf + evals * sub_n) * per_doc_fwd
    if max_minutes: steps = min(steps, max(10, int((max_minutes * 60 * 0.92 - eval_s) / sec_step)))
    print(f"[{name}] {fam} | technique {technique}{' (method ' + method + ')' if kd else ''} | L={L} W={W} p_tail={p_tail} | {n_params/1e6:.0f}M params, {n_train/1e6:.2f}M trained ({n_train/n_params:.1%}) | lr {lr:g} (head {head_lr:g}) | EMA {ema_decay} | {docs_per_step} cases/step\n"
          f"   plan: {steps} steps = {steps*docs_per_step} cases = {steps*docs_per_step/len(tr_idx):.2f} epochs | {sec_step:.2f}s/step -> ~{steps*sec_step/60:.0f} min training + ~{eval_s/60:.1f} min inference")
    perm, ptr, evals_at, run_loss, hist = rng.permutation(tr_idx), 0, {int(round(steps * (i + 1) / evals)) for i in range(evals)}, 0.0, []
    for step in range(1, steps + 1):
        w = min(step / max(1, int(warmup * steps)), (steps - step + 1) / max(1, steps - int(warmup * steps)), 1.0)
        for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * w
        for _ in range(accum):
            if ptr + mb > len(perm): perm, ptr = rng.permutation(tr_idx), 0
            loss = micro(perm[ptr:ptr + mb]); ptr += mb; scaler.scale(loss / accum).backward(); run_loss += loss.item() / accum
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); ema.update(step)
        if step % max(1, steps // 12) == 0: print(f"   step {step}/{steps} | train loss {run_loss/max(1, steps//12):.4f} | {(time.time()-t_start)/60:.1f} min", flush=True); run_loss = 0.0
        if step in evals_at:                                                       # monitoring only: EMA weights, last window, dev subset (no snapshot selection: the final EMA weights are used)
            ema.apply(); z = predict_windows(model, sub_tails, L, 1)[:, 0]; ema.restore(); model.train(); a_ = roc_auc(y_sub, z); t_, f_ = best_threshold(y_sub, sigmoid(z)); hist.append((step, round(a_, 4), round(f_, 4)))
            print(f"   [dev subset {sub_n}, EMA, last window] step {step}: AUC {a_:.4f} | macro-F1 {f_:.4f}", flush=True)
    ema.apply()                                                                    # final model = EMA weights
    Zd, Zt = predict_windows(model, dv_tails, L, W_inf), predict_windows(model, te_tails, L, W_inf)            # training samples from W windows; inference scores W_infer (default: the same)
    aucs = {a: roc_auc(y_dev, aggregate(Zd, a)) for a in AGG_GRID}; a_best = max(AGG_GRID, key=lambda a: (round(aucs[a], 6), -a)); m_dev, m_te = aggregate(Zd, a_best), aggregate(Zt, a_best); p_dev, p_te = sigmoid(m_dev), sigmoid(m_te)
    thr, f_tuned = best_threshold(y_dev, p_dev); at05 = prf(y_dev, (p_dev >= 0.5).astype(int)); rm = prf(y_dev, rm_predict(p_dev, prior)); tuned = prf(y_dev, (p_dev >= thr).astype(int))
    pred_te = rm_predict(p_te, prior) if policy == "rate_matched" else (p_te >= thr).astype(int)
    res = dict(W_infer=W_inf, name=name, model=model_key, family=fam, method=method, technique=technique, kd=bool(kd), n_layers=n_layers or 0, control=bool(control), L=L, W=W, p_tail=p_tail, pool=pool, params_M=round(n_params / 1e6, 1), trainable_M=round(n_train / 1e6, 3), lr=lr, epochs=round(steps * docs_per_step / len(tr_idx), 2), steps=steps, agg_a=a_best,
               dev_auc=aucs[a_best], dev_auc_last_window=aucs[0.0], dev_f1_ratematched=rm["macro_f1"], dev_f1_tuned=f_tuned, dev_f1_at_0p5=at05["macro_f1"], threshold=thr, dev_p1=tuned["p1"], dev_r1=tuned["r1"], dev_cases=len(y_dev),
               test_pos_rate_at_dev_thr=float(np.mean(p_te >= thr)), policy=policy, minutes=round((time.time() - t_start) / 60, 1), history=str(hist))
    print(f"[{name}] DEV: AUC {aucs[a_best]:.4f} (last window only: {aucs[0.0]:.4f}; chosen recency decay a={a_best}) | macro-F1 rate-matched {rm['macro_f1']:.4f} | tuned {f_tuned:.4f} (thr {thr:.2f}) | at 0.5 {at05['macro_f1']:.4f} | {res['minutes']} min\n"
          f"   test: share predicted positive at the dev threshold = {res['test_pos_rate_at_dev_thr']:.3f} (dev positive rate {prior:.3f}) -> policy '{policy}' used for the submission file")
    if write:
        rd = WORK / "runs" / name; rd.mkdir(exist_ok=True); np.save(rd / "dev_Z.npy", Zd); np.save(rd / "test_Z.npy", Zt); np.save(rd / "dev_probs.npy", p_dev); np.save(rd / "test_probs.npy", p_te); json.dump(res, open(rd / "result.json", "w"), indent=1)
        if save_model: torch.save({"state": model.state_dict()}, rd / "model.pt"); json.dump(dict(model_key=model_key, method=method, hp=hp, n_layers=n_layers, pool=pool, L=L, W=W, agg_a=a_best), open(rd / "meta.json", "w"))
        rids = [te["rids"][i] for i in te_idx]; write_submission(SUBDIR / f"{name}.json", rids, pred_te); validate_submission(SUBDIR / f"{name}.json", rids)
        row = pd.DataFrame([res]); led = pd.concat([pd.read_csv(LEDGER).query('name != @name'), row]) if LEDGER.exists() else row; led.to_csv(LEDGER, index=False)
        print(f"   saved {SUBDIR / (name + '.json')} ({len(rids)} predictions, format validated) and the ledger row")
    del model, opt, bb, ema; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return res

def lr_pilot(model_key, grid=(1.5e-5, 3e-5, 1e-4), minutes=10, L=512, **kw):
    """short runs on a subset; picks the learning rate by dev AUC (threshold-free, less noisy than F1). Differences below about 0.01 AUC are noise: prefer the middle value then."""
    out = {}
    for lr_ in grid:
        r = run_experiment(f"pilot_{model_key}_{lr_:g}", model_key, L=L, W=1, epochs=99, max_minutes=minutes, lr=lr_, max_train_docs=10000, dev_cap=1500, test_cap=50, evals=1, write=False, ema_decay=0.99, **kw); out[lr_] = r["dev_auc"]
    best = max(out, key=out.get); print("\nLR pilot (dev AUC):", {f"{k:g}": round(v, 4) for k, v in out.items()}, "-> best", f"{best:g}"); return best

def load_finetuned(run_name):
    """rebuild a fine-tuned model from runs/<name>/model.pt + meta.json (written when save_model=True)"""
    rd = WORK / "runs" / run_name; meta = json.load(open(rd / "meta.json")); ck = torch.load(rd / "model.pt", map_location="cpu", weights_only=False)
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS[meta["model_key"]])
    if meta.get("n_layers"): truncate_layers(bb, meta["n_layers"])
    model = DocClassifier(bb, hidden, causal, 1, 0.0, meta.get("pool", "last")); apply_method(model, meta["method"], meta.get("hp")); model.load_state_dict(ck["state"]); return model.to(device).eval(), meta

@torch.no_grad()
def make_teacher(run_names, out=None):
    """teacher for distillation: the mean margin (logit1 - logit0, all windows, the run's own recency decay) of finished runs on EVERY training case. Costs one inference pass over the training set per run (about 15-30 min)."""
    tr = DATA["train"]; margins = []
    for name in run_names:
        model, meta = load_finetuned(name); t0 = time.time(); Z = predict_windows(model, tr["tails"], meta["L"], meta["W"]); m = aggregate(Z, meta["agg_a"]); margins.append(m)
        print(f"teacher {name}: training-set accuracy {((m > 0) == (tr['labels'] == 1)).mean():.3f} (high is expected: it was trained on these cases) | {(time.time()-t0)/60:.1f} min")
        del model; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    tm = np.mean(margins, 0).astype(np.float32); path = Path(out) if out else WORK / "teacher_margin.npy"; np.save(path, tm)
    print(f"teacher = mean margin of {run_names} -> {path} | margin std {tm.std():.2f} (distillation temperature T softens it)"); return str(path)

# ---- run only what can run: experiments whose Phase 1 model is missing are skipped (with a message), everything else goes on ----
def run_or_skip(name, model_key, **kw):
    """run an experiment if its Phase 1 model is available (re-scanning /kaggle/input first); otherwise print why and carry on"""
    if model_key not in MODEL_PATHS: refresh_models(verbose=False)
    if model_key not in MODEL_PATHS:
        print(f"SKIPPED {name}: the Phase 1 '{model_key}' model is not available yet (found: {sorted(MODEL_PATHS)}). Attach it or finish its pre-training, then re-run this cell."); return None
    return run_experiment(name, model_key, **kw)
def pilot_or_skip(model_key, default_lr, **kw):
    if model_key not in MODEL_PATHS: refresh_models(verbose=False)
    if model_key not in MODEL_PATHS:
        print(f"SKIPPED the learning-rate pilot for '{model_key}' (model not available); the default {default_lr:g} is used if you run its experiment later."); return default_lr
    return lr_pilot(model_key, **kw)
def show_plan():
    plan = [("P1_gpt2_full", "gpt2_new", "full fine-tuning"), ("P2_enc_full", "modernbert", "full fine-tuning"), ("P3_enc_lorma", "modernbert", "LoRMA+ (compulsory)"), ("P4_t5_lora", "t5", "LoRA"),
            ("P5_bilstm_partial", "bilstm", "partial fine-tuning"), ("P6_gpt2_ia3", "gpt2_new", "IA3 (optional)"), ("P7_gpt2_6L_kd", "gpt2_new", "knowledge distillation (needs a saved teacher from P1 and/or P2)")]
    refresh_models(verbose=False); done = set(pd.read_csv(LEDGER).name) if LEDGER.exists() else set()
    print("Phase 1 models available:", sorted(MODEL_PATHS), "| MISSING:", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k not in MODEL_PATHS] or "none")
    print(f"{'run':20s} {'model':11s} {'technique':62s} status")
    for n, m, tech in plan:
        teach = (WORK / "runs" / "P1_gpt2_full" / "model.pt").exists() or (WORK / "runs" / "P2_enc_full" / "model.pt").exists()
        st = "done" if n in done else ("SKIPPED: model missing" if m not in MODEL_PATHS else ("will run once P1 or P2 has finished" if (n == "P7_gpt2_6L_kd" and not teach) else "will run"))
        print(f"{n:20s} {m:11s} {tech:62s} {st}")


In [ ]:
# ================= RESTORE EARLIER RESULTS  [NEEDS your earlier Phase 4 notebook output attached (Add Input > Notebook Output Files): it holds the finished GPT-2 and encoder runs with their saved models and per-window scores. A backup dataset works too.] =================
# Merges every source it finds (later sources win on a file name): the runs (per-window scores, probabilities, saved models), the submissions and the ledger (rows merged by name).
outs = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/cjpe/ledger.csv", recursive=True)}); mfs = [m for m in glob.glob("/kaggle/input/**/MANIFEST.json", recursive=True) if re.search(r"phase4|cjpe", m, re.I)]
assert outs or mfs, "attach your earlier Phase 4 notebook output (Add Input > Notebook Output Files) or the backup dataset first"
ledgers = []
for src in outs:
    src = Path(src)
    for sub in ("runs", "submissions"):
        if (src / sub).exists(): shutil.copytree(src / sub, WORK / sub, dirs_exist_ok=True)
    ledgers.append(pd.read_csv(src / "ledger.csv")); print("restored from the notebook output", src)
for mf in mfs:                                                                     # backup datasets: flat file names <run>__<file>, submission__<file>, top__ledger.csv
    src = Path(mf).parent; man = json.load(open(mf)); bad = []
    for name, meta in man["files"].items():
        f = src / name
        if not f.exists() or f.stat().st_size != meta["bytes"]: bad.append(name); continue
        run, fn = name.split("__", 1)
        if run == "top" and fn == "ledger.csv": ledgers.append(pd.read_csv(f)); continue
        dst = SUBDIR / fn if run == "submission" else Path("/kaggle/working") / fn if run == "top" else WORK / "runs" / run / fn
        if dst.exists(): continue
        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst)
    print(f"restored from the backup dataset {src}" + (f" | missing or wrong size: {bad}" if bad else " | all sizes match the manifest"))
if ledgers: pd.concat(ledgers).drop_duplicates("name", keep="last").to_csv(LEDGER, index=False)
have = sorted(p.parent.name for p in (WORK / "runs").glob("*/dev_Z.npy")); saved = sorted(p.parent.name for p in (WORK / "runs").glob("*/model.pt"))
print(f"ledger rows: {len(pd.read_csv(LEDGER))} | runs with per-window scores: {have} | runs with a saved model (usable as distillation teachers): {saved}")


In [ ]:
# ================= RESILIENCE HELPER + SHARED SETTINGS  [pure code; run it before the experiment cells] =================
# A failing experiment prints its traceback and the notebook carries on, so one crash cannot stop 'Save & Run All' before the later cells.
import traceback
def safe(label, fn):
    try: return fn()
    except Exception:
        print(f"{label} FAILED, continuing with the next cell:"); traceback.print_exc(); gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return None
# Which NEW runs happen in this session. P3, P4, P5 and P7 are the experiments the capstone rules still need (LoRMA, T5, BiLSTM, distillation); P6 (IA3) and X1 (a second encoder seed) are optional ensemble members.
# Rough times on a T4: P3 ~60 min, P4 ~10 min (the probe) up to ~55 min, P5 ~75 min, P7 ~120 min (two teacher passes over the training cases, the student, its control), X1 ~55 min, P6 ~60 min. Everything: about 6-7 hours.
RUN = {"P3", "P4", "P5", "P7", "X1"}
W_MAIN, W_INFER = 4, 8            # windows of 512 tokens: training samples from 4, inference scores up to 8 (the cache holds the last 4,096 tokens); only longer cases get more than 4
PRIOR = float(np.mean(DATA["dev"]["labels"] == 1)); MEMBER_REL = 0.90
print(f"new runs selected: {sorted(RUN)} | training windows {W_MAIN}, inference windows {W_INFER} | dev positive rate {PRIOR:.3f}")


In [ ]:
# ================= P3: ENCODER, LoRMA+ (COMPULSORY)  [NEEDS the encoder Phase 1 checkpoint + a GPU; ~60 min] =================
# Multiplicative low-rank adaptation of q and v, the backbone frozen (1.5% of the parameters train). Learning rate 5e-4: the best of 2e-4 / 5e-4 / 1.5e-3 in the Phase 5 pilot of the same encoder (not re-piloted here). 4 epochs: adapters converge more slowly.
if "P3" in RUN: P3 = safe("P3", lambda: run_or_skip("P3_enc_lorma", "modernbert", method="lorma_plus", hp=dict(r=16, alpha=32), L=512, W=W_MAIN, W_infer=W_INFER, epochs=4, lr=5e-4, p_tail=0.5))
else: print("P3 not selected (RUN)")


In [ ]:
# ================= P4: ENCODER-DECODER (T5 encoder)  [NEEDS the T5 Phase 1 checkpoint + a GPU; probe ~6 min per method, then ~55-60 min] =================
# In earlier Phase 4 and Phase 5 runs T5 learned little or nothing (a Phase 4 full fine-tuning probe: dev AUC 0.51). T5 is known to be unstable in fp16, so this cell works in fp32 and probes first: LoRA, then full fine-tuning; a full run starts only for a probe whose dev AUC beats 0.56 (my threshold).
# If neither probe learns, a time-capped LoRA run (45 minutes) is still made so that the capstone rule 'an experiment on every Phase 1 architecture' is met: it will be weak, it is excluded from the ensemble, and the final report says so.
if "P4" in RUN:
    _amp = AMP; AMP = False; P4 = None
    try:
        for tag, method, hp_, lr_, ep_ in (("lora", "lora", dict(r=16, alpha=32), 5e-4, 4), ("full", "full", None, 5e-5, 3)):
            chk = run_or_skip(f"probe_t5_{tag}_fp32", "t5", method=method, hp=hp_, L=512, W=1, epochs=99, max_minutes=6, lr=lr_, max_train_docs=8000, dev_cap=1000, test_cap=20, evals=1, write=False, ema_decay=0.99)
            if chk is None: break
            if chk["dev_auc"] > 0.56:
                print(f"T5 {tag} probe learns (dev AUC {chk['dev_auc']:.3f}): starting the full run in fp32")
                P4 = safe("P4", lambda: run_or_skip(f"P4_t5_{tag}", "t5", method=method, hp=hp_, L=512, W=W_MAIN, W_infer=W_INFER, epochs=ep_, lr=lr_, p_tail=0.5)); break
            print(f"T5 {tag} probe stayed near chance (dev AUC {chk['dev_auc']:.3f})")
        else:
            print("T5 did not learn in either probe: a time-capped LoRA run is made only to cover the architecture (expect a weak result)")
            P4 = safe("P4 capped", lambda: run_or_skip("P4_t5_lora", "t5", method="lora", hp=dict(r=16, alpha=32), L=512, W=W_MAIN, W_infer=W_INFER, epochs=2, lr=5e-4, p_tail=0.5, max_minutes=45))
    except Exception: print("T5 cell FAILED, continuing with the next cell:"); traceback.print_exc(); gc.collect()
    finally: AMP = _amp
else: print("P4 not selected (RUN)")


In [ ]:
# ================= P5: RECURRENT (BiLSTM), PARTIAL FINE-TUNING  [NEEDS the BiLSTM Phase 1 checkpoint + a GPU; ~75 min] =================
# The embeddings and the lowest LSTM layer are frozen, the rest trains; the BiLSTM has no attention layers, so adapters do not apply to it. 512-token windows and 2 epochs so that it fits (the BiLSTM runs at about 9k tokens/s).
# Skipped with a message while the BiLSTM Phase 1 dataset is not attached; then the capstone rule check below reports that architecture as not covered.
if "P5" in RUN: P5 = safe("P5", lambda: run_or_skip("P5_bilstm_partial", "bilstm", method="partial", hp=dict(frac=0.5), L=512, W=W_MAIN, W_infer=W_INFER, epochs=2, lr=DEFAULT_LR["bilstm"], p_tail=0.5))
else: print("P5 not selected (RUN)")


In [ ]:
# ================= P6 (OPTIONAL): DECODER (GPT-2), IA3  [NEEDS the GPT-2 Phase 1 checkpoint + a GPU; ~60 min] =================
# Learned rescaling of keys, values and FFN activations (the smallest adapter): a sixth technique family and one more diverse ensemble member. Add "P6" to RUN in the settings cell to run it.
if "P6" in RUN: P6 = safe("P6", lambda: run_or_skip("P6_gpt2_ia3", "gpt2_new", method="ia3", L=512, W=W_MAIN, W_infer=W_INFER, epochs=4, lr=ADAPTER_LR["ia3"], p_tail=0.5))
else: print("P6 not selected (RUN)")


In [ ]:
# ================= P7: KNOWLEDGE DISTILLATION  [NEEDS the saved models of P1 and/or P2 (restored) + the GPT-2 Phase 1 checkpoint + a GPU; teachers ~35 min, student ~35 min, control ~35 min] =================
# Teacher = the mean margin of the saved GPT-2 (P1) and encoder (P2) runs on every training case. Student = the first 6 layers of the Phase 1 GPT-2. Loss = alpha * cross-entropy + (1 - alpha) * T^2 * KL(teacher || student), on the sampled window of each case.
# P7c is the SAME student without distillation, so the benefit is measured (an ablation, never counted). If only one teacher exists it is used alone; if none exists the cell is skipped.
if "P7" in RUN:
    TEACH = [n for n in ("P1_gpt2_full", "P2_enc_full") if (WORK / "runs" / n / "model.pt").exists()]
    if not TEACH: print("SKIPPED P7 (distillation): no saved teacher model: the restore cell must bring back P1 and/or P2 with their model.pt")
    elif "gpt2_new" not in MODEL_PATHS and "gpt2_new" not in refresh_models(verbose=False): print("SKIPPED P7 (distillation): the student is a half-depth GPT-2 and the GPT-2 Phase 1 checkpoint is not available")
    else:
        print("teacher runs:", TEACH, "(single teacher)" if len(TEACH) == 1 else "(ensemble)"); TEACHER = make_teacher(TEACH)
        P7 = safe("P7", lambda: run_or_skip("P7_gpt2_6L_kd", "gpt2_new", method="full", n_layers=6, kd=dict(path=TEACHER, T=3.0, alpha=0.5), L=512, W=W_MAIN, W_infer=W_INFER, epochs=3, lr=5e-5, p_tail=0.5))
        P7c = safe("P7c", lambda: run_or_skip("P7c_gpt2_6L_nokd", "gpt2_new", method="full", n_layers=6, L=512, W=W_MAIN, W_infer=W_INFER, epochs=3, lr=5e-5, p_tail=0.5, control=True))
else: print("P7 not selected (RUN)")


In [ ]:
# ================= X1 (OPTIONAL ENSEMBLE MEMBER): ENCODER, full fine-tuning, a SECOND SEED  [NEEDS the encoder Phase 1 checkpoint + a GPU; ~55 min] =================
# Same recipe as P2 with another seed: an ablation (control=True, never a counted experiment) that makes the ensemble less dependent on one training run. P2's validation curve peaked around epoch 2-3, so no extra epochs.
if "X1" in RUN: X1 = safe("X1", lambda: run_or_skip("X1_enc_full_s7", "modernbert", method="full", L=512, W=W_MAIN, W_infer=W_INFER, epochs=3, lr=5e-5, seed=7, p_tail=0.5, control=True))
else: print("X1 not selected (RUN)")


In [ ]:
# ================= EIGHT WINDOWS FOR THE RESTORED MODELS  [NEEDS the Phase 1 checkpoint of each model + its saved model.pt (restored); ~3 min per model] =================
# Re-scores the finished GPT-2 and encoder runs with up to 8 windows (only cases longer than 2,048 tokens get more than 4) and saves '<name>_W8' as a candidate. Skipped if that file already exists (your earlier run may have made P2_enc_full_W8).
for name in ("P1_gpt2_full", "P2_enc_full"):
    new = f"{name}_W8"
    if (WORK / "runs" / new / "result.json").exists(): print(f"{new} already exists (restored): skipped"); continue
    if not (WORK / "runs" / name / "model.pt").exists(): print(f"SKIPPED {name}: no saved model"); continue
    def one(name=name, new=new):
        model, meta = load_finetuned(name); r0 = json.load(open(WORK / "runs" / name / "result.json")); y_dev = DATA["dev"]["labels"]; t0 = time.time()
        Zd, Zt = predict_windows(model, DATA["dev"]["tails"], meta["L"], 8), predict_windows(model, DATA["test"]["tails"], meta["L"], 8)
        aucs = {a: roc_auc(y_dev, aggregate(Zd, a)) for a in AGG_GRID}; a_b = max(AGG_GRID, key=lambda a: (round(aucs[a], 6), -a)); p_dev, p_te = sigmoid(aggregate(Zd, a_b)), sigmoid(aggregate(Zt, a_b)); thr, f_t = best_threshold(y_dev, p_dev); rm = prf(y_dev, rm_predict(p_dev, PRIOR))
        res = dict(r0); res.update(name=new, W_infer=8, agg_a=a_b, dev_auc=aucs[a_b], dev_auc_last_window=aucs[0.0], dev_f1_ratematched=rm["macro_f1"], dev_f1_tuned=f_t, threshold=thr, dev_cases=len(y_dev), test_pos_rate_at_dev_thr=float(np.mean(p_te >= thr)))
        nd = WORK / "runs" / new; nd.mkdir(exist_ok=True); np.save(nd / "dev_Z.npy", Zd); np.save(nd / "test_Z.npy", Zt); np.save(nd / "dev_probs.npy", p_dev); np.save(nd / "test_probs.npy", p_te); json.dump(res, open(nd / "result.json", "w"), indent=1)
        rids = DATA["test"]["rids"]; write_submission(SUBDIR / f"{new}.json", rids, rm_predict(p_te, PRIOR)); validate_submission(SUBDIR / f"{new}.json", rids)
        pd.concat([pd.read_csv(LEDGER).query("name != @new"), pd.DataFrame([res])]).to_csv(LEDGER, index=False)
        print(f"{name}: 4 -> 8 windows: dev AUC {r0['dev_auc']:.4f} -> {aucs[a_b]:.4f} | macro-F1 rate-matched {r0['dev_f1_ratematched']:.4f} -> {rm['macro_f1']:.4f} | {(time.time()-t0)/60:.1f} min | saved as candidate '{new}'")
        del model; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    safe(f"8 windows {name}", one)


In [ ]:
# ================= STACKER TOOLS  [pure code; needs scikit-learn] =================
# A stacker is trained on the VALIDATION split from the models' per-window scores and scored OUT-OF-FOLD (5-fold cross-validation over validation cases): the score it reports is for cases it was not trained on, and its test scores are the average of the 5 fold models.
# The regularisation strength is chosen from 5 values by the out-of-fold AUC. No test label is used anywhere (test labels are private).
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
def paired_boot_binary(y, pa, pb, n=300, seed=0):
    """paired bootstrap over cases of (macro-F1 of predictions A) - (macro-F1 of predictions B): mean difference, 95% interval, share of resamples where A wins"""
    r = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); d.append(prf(y[i], pa[i])["macro_f1"] - prf(y[i], pb[i])["macro_f1"])
    d = np.array(d); return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float((d > 0).mean())
def feat_member(Z, a=0.75, width=8):
    """12 features of one model's per-window margins [cases, windows] (NaN = the case has no such window): the 8 window margins (0 if missing), mean / max / min over the available windows, and the recency-weighted mean"""
    n = Z.shape[0]; Zp = np.full((n, width), np.nan, np.float32); k = min(width, Z.shape[1]); Zp[:, :k] = Z[:, :k]; ok = ~np.isnan(Zp); Z0 = np.where(ok, Zp, 0.0); cnt = np.maximum(ok.sum(1), 1)
    w = a ** np.arange(width); rec = (Z0 * w * ok).sum(1) / np.maximum((w * ok).sum(1), 1e-9)
    return np.concatenate([Z0, (Z0.sum(1) / cnt)[:, None], np.where(ok, Zp, -np.inf).max(1)[:, None], np.where(ok, Zp, np.inf).min(1)[:, None], rec[:, None]], 1).astype(np.float32)
def stack_lr_cv(Xd, y, Xt, K=5, Cs=(0.01, 0.03, 0.1, 0.3, 1.0), seed=0):
    folds = np.array_split(np.random.default_rng(seed).permutation(len(y)), K); res = {}
    for C in Cs:
        oof = np.zeros(len(y)); te = np.zeros(len(Xt))
        for fi in range(K):
            va = folds[fi]; tr = np.concatenate([folds[k] for k in range(K) if k != fi]); clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=500)).fit(Xd[tr], y[tr])
            oof[va] = clf.decision_function(Xd[va]); te += clf.decision_function(Xt) / K
        res[C] = (roc_auc(y, oof), oof, te)
    best = max(res, key=lambda c: res[c][0]); return res[best][1], res[best][2], best, {c: round(v[0], 4) for c, v in res.items()}


In [ ]:
# ================= LONG-DOCUMENT REPRESENTATION STUDY  [RUNS NOW from the saved per-window scores; CPU, seconds] =================
# How much does reading more of a case help? For every finished run: the last 512 tokens only, the last 4 windows (2,048 tokens), and all windows (up to 8 = 4,096 tokens), each with its best recency decay,
# scored on the whole dev split, with a paired bootstrap of the macro-F1 (rate-matched predictions) between neighbouring representations. This is the 'approaches to representing long documents' comparison of the capstone.
y_dev = DATA["dev"]["labels"]; study_rows = []
for rd in sorted((WORK / "runs").glob("*")):
    if rd.name.startswith(("pilot", "probe", "smoke")) or not all((rd / f).exists() for f in ("result.json", "dev_Z.npy")): continue
    r = json.load(open(rd / "result.json")); Zd = np.load(rd / "dev_Z.npy")
    if r.get("dev_cases") != len(y_dev): continue
    reps = {"last window (512 tokens)": Zd[:, :1], "last 4 windows (2,048)": Zd[:, :4]}
    if Zd.shape[1] > 4: reps[f"all {Zd.shape[1]} windows (up to 4,096)"] = Zd
    prev = None
    for label, Z in reps.items():
        a = max(AGG_GRID, key=lambda a_: (round(roc_auc(y_dev, aggregate(Z, a_)), 6), -a_)); s = aggregate(Z, a); pred = rm_predict(s, PRIOR); auc = roc_auc(y_dev, s); f1 = prf(y_dev, pred)["macro_f1"]; diff = ""
        if prev is not None: d, lo, hi, w = paired_boot_binary(y_dev, pred, prev, n=200); diff = f"{d:+.4f} [{lo:+.4f}, {hi:+.4f}]"
        study_rows.append(dict(run=rd.name, representation=label, dev_auc=round(auc, 4), macro_f1_ratematched=round(f1, 4), vs_previous_row=diff)); prev = pred
if study_rows: print(pd.DataFrame(study_rows).to_string(index=False)); pd.DataFrame(study_rows).to_csv(WORK / "representation_study.csv", index=False)
else: print("no run with per-window scores yet")


In [ ]:
# ================= CANDIDATES, STACKERS, THE FINAL prediction.json, AND THE CAPSTONE RULE CHECK  [RUNS after the restore (and whichever new runs finished); trains only small logistic stackers] =================
ADOPT_GAIN, OUT_DIR = 0.003, Path("/kaggle/working"); y_dev = DATA["dev"]["labels"]; rids = DATA["test"]["rids"]; lens_d, lens_t = DATA["dev"]["lens"], DATA["test"]["lens"]
info = {}
for rd in sorted((WORK / "runs").glob("*")):
    if rd.name.startswith(("pilot", "probe", "smoke")) or not all((rd / f).exists() for f in ("result.json", "dev_Z.npy", "test_Z.npy")): continue
    r = json.load(open(rd / "result.json"))
    if r.get("dev_cases") == len(y_dev): info[rd.name] = r
assert info, "no usable run: run the restore cell first"
base_of = lambda n: re.sub(r"_W8$", "", n); best_variant = {}
for n, r in info.items():                                                         # ONE variant per checkpoint: the 8-window re-score replaces its own 4-window run (they are the same model)
    b = base_of(n)
    if b not in best_variant or r["dev_auc"] > info[best_variant[b]]["dev_auc"]: best_variant[b] = n
cand_names = sorted(best_variant.values(), key=lambda n: -info[n]["dev_auc"]); best_auc = info[cand_names[0]]["dev_auc"]; MEMBER_REL = globals().get("MEMBER_REL", 0.90)
members = [n for n in cand_names if info[n]["dev_auc"] >= max(0.55, MEMBER_REL * best_auc)]
print("CANDIDATE MODELS (one variant per checkpoint; a MEMBER has dev AUC >= " + f"{max(0.55, MEMBER_REL * best_auc):.3f}" + "):")
print(pd.DataFrame([dict(name=n, model=info[n]["model"], technique=info[n]["technique"], control=info[n]["control"], epochs=info[n]["epochs"], dev_auc=round(info[n]["dev_auc"], 4), last_window_auc=round(info[n]["dev_auc_last_window"], 4), macro_f1_ratematched=round(info[n]["dev_f1_ratematched"], 4), member=n in members) for n in cand_names]).to_string(index=False))
Zd, Zt, Md, Mt = {}, {}, {}, {}
for n in members:
    rd = WORK / "runs" / n; Zd[n], Zt[n] = np.load(rd / "dev_Z.npy"), np.load(rd / "test_Z.npy"); a = info[n]["agg_a"]; md, mt = aggregate(Zd[n], a), aggregate(Zt[n], a); s = md.std() + 1e-9; Md[n], Mt[n] = md / s, mt / s
def evaluate_scores(s_dev, s_te):
    pd_, pt_ = rm_predict(s_dev, PRIOR), rm_predict(s_te, PRIOR); m = prf(y_dev, pd_); thr, f_t = best_threshold(y_dev, sigmoid(s_dev))
    return dict(s_dev=s_dev, s_te=s_te, pred_dev=pd_, pred_te=pt_, auc=roc_auc(y_dev, s_dev), f1=m["macro_f1"], f1_tuned=f_t, macro_p=m["macro_p"], macro_r=m["macro_r"], p1=m["p1"], r1=m["r1"])
C = {"best_single": dict(what=members[0], **evaluate_scores(Md[members[0]], Mt[members[0]]))}
aucs_k = [roc_auc(y_dev, np.mean([Md[n] for n in members[:k]], 0)) for k in range(1, len(members) + 1)]; k_ens = max(k for k, a in enumerate(aucs_k, 1) if a >= max(aucs_k) - 0.002); ens = members[:k_ens]
C["mean_ensemble"] = dict(what=f"mean of {ens}", **evaluate_scores(np.mean([Md[n] for n in ens], 0), np.mean([Mt[n] for n in ens], 0))); print("\nensemble dev AUC by number of models (best first):", dict(zip(members, [round(a, 4) for a in aucs_k])), "-> using", ens)
if len(members) >= 2:
    o1, t1, c1, a1 = stack_lr_cv(np.stack([Md[n] for n in members], 1), y_dev, np.stack([Mt[n] for n in members], 1)); C["S1_model_weights"] = dict(what=f"logistic weights over {len(members)} models (C={c1})", **evaluate_scores(o1, t1))
def glob_feats(lens, Zs): return np.stack([np.log1p(lens), np.max([(~np.isnan(z)).sum(1) for z in Zs], 0) / 8.0], 1)
Xd = np.concatenate([feat_member(Zd[n], info[n]["agg_a"]) for n in members] + [glob_feats(lens_d, [Zd[n] for n in members])], 1); Xt = np.concatenate([feat_member(Zt[n], info[n]["agg_a"]) for n in members] + [glob_feats(lens_t, [Zt[n] for n in members])], 1)
o2, t2, c2, a2 = stack_lr_cv(Xd, y_dev, Xt); C["S2_window_stacker"] = dict(what=f"logistic regression over every window of {len(members)} models + case length ({Xd.shape[1]} features, C={c2})", **evaluate_scores(o2, t2))
ref = "mean_ensemble" if C["mean_ensemble"]["auc"] >= C["best_single"]["auc"] + ADOPT_GAIN else "best_single"; chosen = ref
for nm in [k for k in C if k.startswith("S")]:
    d, lo, hi, w = paired_boot_binary(y_dev, C[nm]["pred_dev"], C[ref]["pred_dev"], n=300); C[nm]["vs_ref"] = (d, lo, hi, w)
    if C[nm]["f1"] >= C[chosen]["f1"] + ADOPT_GAIN and C[nm]["auc"] >= C[ref]["auc"] + 0.002 and lo > 0 and C[nm]["f1"] > C[chosen]["f1"]: chosen = nm
tab = []
for k, c in C.items():
    d = c.get("vs_ref"); tab.append(dict(candidate=k, dev_auc=round(c["auc"], 4), macro_f1_ratematched=round(c["f1"], 4), macro_f1_tuned=round(c["f1_tuned"], 4), macro_precision=round(c["macro_p"], 4), macro_recall=round(c["macro_r"], 4), **({"vs_" + ref: f"{d[0]:+.4f} [{d[1]:+.4f}, {d[2]:+.4f}]"} if d else {})))
print("\nVALIDATION COMPARISON (stackers: OUT-OF-FOLD scores; predictions are rate-matched to the dev positive rate " + f"{PRIOR:.2f}" + "; the paired bootstrap compares a stacker with the reference '" + ref + "'):"); print(pd.DataFrame(tab).to_string(index=False))
print(f"\nDECISION RULE: the reference is the mean ensemble only if its AUC beats the best single run by {ADOPT_GAIN}; a stacker replaces the reference only if its macro-F1 wins by {ADOPT_GAIN}, its AUC by 0.002, AND its paired 95% interval is above 0.\n=> prediction.json = {chosen}: {C[chosen]['what']}")
for k, c in C.items(): write_submission(OUT_DIR / f"prediction_{k}.json", rids, c["pred_te"]); validate_submission(OUT_DIR / f"prediction_{k}.json", rids)
write_submission(OUT_DIR / "prediction.json", rids, C[chosen]["pred_te"]); validate_submission(OUT_DIR / "prediction.json", rids); c = C[chosen]; rng = np.random.default_rng(0)
boot = [prf(y_dev[i], c["pred_dev"][i])["macro_f1"] for i in (rng.integers(0, len(y_dev), len(y_dev)) for _ in range(300))]; lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"\nFINAL (dev, {len(y_dev)} cases): AUC {c['auc']:.4f} | macro-F1 {c['f1']:.4f} (95% bootstrap interval {lo:.3f} - {hi:.3f}) | macro-precision {c['macro_p']:.4f} | macro-recall {c['macro_r']:.4f} | class-1 precision {c['p1']:.4f} recall {c['r1']:.4f} | tuned-threshold macro-F1 {c['f1_tuned']:.4f}")
print(f"test: {int(c['pred_te'].sum())} of {len(rids)} flagged 1 ({c['pred_te'].mean():.1%}; rate matching uses the dev positive rate and no test labels) | file validated: {len(rids)} ids, integer 0/1. One standard error on {len(rids)} test cases is about 0.012: differences under ~0.02 are noise.")
q = np.digitize(lens_d, np.quantile(lens_d, [0.25, 0.5, 0.75])); print("dev macro-F1 by case length (token quartiles):", {f"Q{k+1} (<= {int(np.quantile(lens_d, (k+1)/4))})": round(prf(y_dev[q == k], c["pred_dev"][q == k])["macro_f1"], 3) for k in range(4)})
# ---------------- the capstone rules over the finished experiments ----------------
import shutil
from itertools import combinations
FAMILY = {"full": "full fine-tuning", "partial": "partial fine-tuning", "lora": "LoRA", "rslora": "LoRA", "lorma_plus": "LoRMA", "lorma_pi": "LoRMA", "ia3": "IA3", "kd": "knowledge distillation"}; REQ_MODELS = ("gpt2_new", "modernbert", "t5", "bilstm")
# An experiment = a distinct (Phase 1 model, technique) pair. Seeds, 8-window re-scores, controls and pilots are not experiments: of runs sharing a pair, only the best is kept.
led = pd.read_csv(LEDGER); led["control"] = led["control"].astype(str).str.lower().eq("true"); led = led[~led.name.str.startswith(("pilot", "probe", "smoke"))]
ex = led[(~led.control) & led.technique.isin(FAMILY) & led.model.isin(REQ_MODELS)].copy(); assert len(ex), "no counted experiment in the ledger"
ex["family"] = ex.technique.map(FAMILY); ex["pair"] = ex.model + "|" + ex.family; ex = ex.sort_values("dev_f1_ratematched", ascending=False).drop_duplicates("pair").reset_index(drop=True)
def rule_check(sub):
    fam = set(sub.family); return dict(all_four_architectures=set(REQ_MODELS) <= set(sub.model), has_lorma="LoRMA" in fam, has_distillation="knowledge distillation" in fam, five_techniques=len(fam) >= 5, six_experiments=len(sub) >= 6, families=sorted(fam))
def choose_set(c, size=6):
    n = min(size, len(c)); best = None
    for combo in combinations(range(len(c)), n):
        sub = c.iloc[list(combo)]; rc = rule_check(sub); strict = all(rc[k] for k in ("all_four_architectures", "has_lorma", "has_distillation", "five_techniques", "six_experiments"))
        key = (strict, len(set(sub.model) & set(REQ_MODELS)), rc["has_lorma"], rc["has_distillation"], min(len(rc["families"]), 5), len(sub), float(sub.dev_f1_ratematched.mean()))
        if best is None or key > best[0]: best = (key, list(sub.name))
    return best
key_, FINAL = choose_set(ex); sel = ex.set_index("name").loc[FINAL].reset_index(); rc = rule_check(sel); strict = key_[0]
unmet = ([] if rc["six_experiments"] else [f"only {len(sel)} distinct experiment(s) (six needed)"]) + ([] if rc["all_four_architectures"] else [f"architectures not covered: {sorted(set(REQ_MODELS) - set(sel.model))}"]) + ([] if rc["has_lorma"] else ["no LoRMA experiment (P3)"]) + ([] if rc["has_distillation"] else ["no distillation experiment (P7)"]) + ([] if rc["five_techniques"] else [f"only {len(rc['families'])} technique families (five needed)"])
status = "ALL CAPSTONE RULES SATISFIED" if strict else "NOT YET COMPLIANT: " + "; ".join(unmet); cols = ["name", "model", "technique", "trainable_M", "dev_auc", "dev_auc_last_window", "dev_f1_ratematched", "dev_f1_tuned"]
print(f"\nSELECTED EXPERIMENTS ({len(sel)}): {status}"); print(sel[cols].round(4).to_string(index=False)); print("technique families:", rc["families"])
weak = [n for n in sel.name if info.get(best_variant.get(base_of(n), n), info.get(n, {})).get("dev_auc", 1) < 0.55]
if weak: print("NOTE: these experiments did not learn (dev AUC below 0.55) and are only counted for the rules, not used in the ensemble:", weak)
FINAL_DIR = WORK / "final_six"; shutil.rmtree(FINAL_DIR, ignore_errors=True); FINAL_DIR.mkdir()
for k_, n_ in enumerate(FINAL, 1):
    if (SUBDIR / f"{n_}.json").exists(): validate_submission(SUBDIR / f"{n_}.json", rids); (FINAL_DIR / f"submission_{k_}_{n_}.json").write_text((SUBDIR / f"{n_}.json").read_text())
def md_table(df):
    try: return df.to_markdown(index=False)
    except Exception: return "```\n" + df.to_string(index=False) + "\n```"
ctl = led[led.control & led.model.isin(REQ_MODELS)]
(FINAL_DIR / "requirements_report.md").write_text("# Phase 4 (CJPE): requirement check\n\n**" + status + "**\n\n" + md_table(sel[cols].round(4)) + f"\n\n- all four Phase 1 architectures: {rc['all_four_architectures']}\n- LoRMA: {rc['has_lorma']}\n- knowledge distillation: {rc['has_distillation']}\n- technique families ({len(rc['families'])}): {', '.join(rc['families'])}\n- the old 3,000-step GPT-2 is not used anywhere\n"
    + (f"\n## Ablations and ensemble members (not counted as experiments)\n{md_table(ctl[['name', 'model', 'technique', 'dev_auc', 'dev_f1_ratematched']].round(4))}\n" if len(ctl) else "") + f"\n## Final prediction\n{chosen}: {C[chosen]['what']}; dev AUC {c['auc']:.4f}, macro-F1 {c['f1']:.4f}\n")
print(f"files: {FINAL_DIR}/ (the selected experiments' submissions + requirements_report.md); prediction.json = {chosen}")


In [ ]:
# ================= BACKUP  [NEEDS Kaggle credentials for the upload, otherwise a dry run; success means the dataset exists and is READY] =================
# Uploads by itself when the credentials cell verified the Kaggle key (KAGGLE_OK); otherwise it is a dry run that only lists what would be uploaded. It never asks you to type anything. A refresh or a new session loses /kaggle/working; this does not.
import hashlib, shutil, subprocess, getpass
DATASET_SLUG = "cs781-phase4-cjpe-final"; INCLUDE_MODELS = True; DRY_RUN = not globals().get("KAGGLE_OK", False); BACKUP_OK = False          # real upload automatically when the credentials cell verified the Kaggle key; otherwise a dry run  # INCLUDE_MODELS=False skips the ~450 MB model files (then distillation teachers are not backed up)
if not DRY_RUN and not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient(); os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
    except Exception:
        raise RuntimeError("Kaggle credentials not found: run the CREDENTIALS cell first, or add the Secrets KAGGLE_USERNAME and KAGGLE_KEY (Add-ons > Secrets, Attached ON). This cell never asks you to type them.")
OWNER = os.environ.get("KAGGLE_USERNAME", "oipranesh"); stage = Path("/kaggle/working/cjpe_backup"); shutil.rmtree(stage, ignore_errors=True); stage.mkdir(parents=True); files = {}
for rd in sorted((WORK / "runs").glob("*")):                                   # flat names (the Kaggle CLI skips sub-folders): <run>__<file>
    if rd.name.startswith(("pilot", "probe", "smoke")): continue
    for f in sorted(rd.iterdir()):
        if f.name == "model.pt" and not INCLUDE_MODELS: continue
        files[f"{rd.name}__{f.name}"] = f
for f in sorted(SUBDIR.glob("*.json")): files[f"submission__{f.name}"] = f
for f in (LEDGER, Path("/kaggle/working/prediction.json"), Path("/kaggle/working/prediction_best_single.json"), Path("/kaggle/working/prediction_ensemble.json")):
    if f.exists(): files[f"top__{f.name}"] = f
assert files, "nothing to back up yet: run an experiment first"
for name, p in files.items():
    try: os.link(p, stage / name)
    except OSError: shutil.copy2(p, stage / name)
def sha256(p, chunk=1 << 24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
runs_in = sorted({n.split("__")[0] for n in files if not n.startswith(("submission__", "top__", "ckpt__"))}); manifest = {"runs": runs_in, "files": {n: {"bytes": (stage / n).stat().st_size, "sha256": sha256(stage / n)} for n in files}}
(stage / "MANIFEST.json").write_text(json.dumps(manifest, indent=1)); (stage / "dataset-metadata.json").write_text(json.dumps({"title": "CS781 Phase 4 CJPE results backup", "id": f"{OWNER}/{DATASET_SLUG}", "licenses": [{"name": "unknown"}]}, indent=1))
tot = sum(p.stat().st_size for p in stage.iterdir()); print(f"staged {len(files) + 2} files, {tot/2**30:.2f} GiB; runs: {runs_in}")
for n in sorted(files): print(f"  {n:58s} {manifest['files'][n]['bytes']/2**20:9.1f} MiB")
kb = shutil.which("kaggle"); cmd_c = ["kaggle", "datasets", "create", "-p", str(stage)]; cmd_v = ["kaggle", "datasets", "version", "-p", str(stage), "-m", f"runs: {', '.join(runs_in)}"]
if DRY_RUN or not kb: print("\nDRY RUN: would run", " ".join(cmd_c), "(or, if the dataset exists)", " ".join(cmd_v), "\nSet DRY_RUN = False and run again to upload.")
else:
    chk = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=180)
    if chk.returncode != 0: print("credentials check FAILED (Internet on? legacy key?):", (chk.stdout + chk.stderr)[:300], "\nNothing was uploaded.")
    else:
        def clean(txt): return "\n".join(l for l in txt.splitlines() if "%|" not in l and l.strip())
        BAD = r"(?i)creation error|slugs? .*null|\bwas not created\b|\bfailed\b|already (exists|in use)|\berror\b"
        r = subprocess.run(cmd_c, capture_output=True, text=True); o = (r.stdout + r.stderr).strip(); print(clean(o)[-900:])
        if r.returncode != 0 or re.search(BAD, o):                                        # the CLI can print an error and still exit 0: do not trust the exit code
            print("-> the create step reported a problem; trying a new VERSION of the dataset (it may already exist)"); r = subprocess.run(cmd_v, capture_output=True, text=True); o2 = (r.stdout + r.stderr).strip(); print(clean(o2)[-900:])
        time.sleep(5); st = subprocess.run(["kaggle", "datasets", "status", f"{OWNER}/{DATASET_SLUG}"], capture_output=True, text=True, timeout=180); st_out = (st.stdout + st.stderr).strip()
        BACKUP_OK = st.returncode == 0 and "ready" in st_out.lower(); print(f"dataset status: {st_out[:100]!r}")
        print(f"\nBACKUP OK (verified: {OWNER}/{DATASET_SLUG} exists and is ready)" if BACKUP_OK else f"\nBACKUP NOT VERIFIED: the dataset does not exist or is not ready. The staged folder {stage} is kept. Part 2 can instead use THIS notebook's output (Add Input > Notebook Output Files), or you can create the dataset from the staged folder by hand.")


In [ ]:
# ================= WHAT 'Save & Run All' KEEPS AS THIS NOTEBOOK'S OUTPUT, a format check, and a size cleanup  [RUNS NOW] =================
DELETE_BULKY = True       # remove the re-downloadable Hugging Face copy and the tokenised cache from the output; predictions, per-window scores, saved models and the ledger are kept
def size_gb(p): return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 2**30 if Path(p).exists() else 0.0
print(f"/kaggle/working before cleanup: {size_gb('/kaggle/working'):.2f} GiB (hf copy {size_gb(WORK / 'hf'):.2f}, cache {size_gb(WORK / 'cache'):.2f}, runs {size_gb(WORK / 'runs'):.2f})")
if DELETE_BULKY: shutil.rmtree(WORK / "hf", ignore_errors=True); shutil.rmtree(WORK / "cache", ignore_errors=True)
if globals().get("BACKUP_OK"): shutil.rmtree("/kaggle/working/cjpe_backup", ignore_errors=True); print("backup verified: the staged copy was removed from the output")
print(f"after cleanup: {size_gb('/kaggle/working'):.2f} GiB\n\nPREDICTION FILES (a JSON list of {{\"id\": ..., \"pred\": 0 or 1}}):")
for f in sorted(SUBDIR.glob("*.json")) + sorted(Path("/kaggle/working").glob("prediction*.json")):
    sub_ = json.load(open(f)); print(f"  {str(f.relative_to('/kaggle/working')):46s} {len(sub_)} predictions | {np.mean([s['pred'] for s in sub_]):.1%} flagged 1")
pj = Path("/kaggle/working/prediction.json")
if pj.exists(): print("\nfirst two objects of prediction.json:", json.load(open(pj))[:2]); validate_submission(pj, DATA["test"]["rids"]); print("format validated against the test ids")
else: print("\nWARNING: /kaggle/working/prediction.json does not exist: the candidates cell did not finish; see the messages above")
